# Credit Risk Prediction using Artificial Neural Network (ANN)
### Fundamentals of Deep Learning | Project-Based Learning

**Student:** Vishal  
**Enrollment Number:** 02018011624

This notebook adds a basic feed-forward Artificial Neural Network to the existing Credit Risk Prediction project. It uses the same UCI German Credit dataset and binary target convention:

- `0` = No Default
- `1` = Default

The network is a genuine neural-network implementation using **PyTorch**. It is an educational extension, not a claim that the original Logistic Regression / Random Forest notebook was already deep learning.

**Architecture:** 48 encoded inputs → Dense(64, ReLU) → Dropout(0.20) → Dense(32, ReLU) → Dropout(0.20) → Dense(1, Logit)

Install dependencies before running:
`pip install pandas numpy scikit-learn matplotlib seaborn torch joblib`

## 1. Import libraries

PyTorch is used to define and train the neural network. Scikit-learn is retained for preprocessing and evaluation.

In [ ]:
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, precision_recall_fscore_support
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Training device:", DEVICE)

## 2. Load the German Credit dataset

The dataset contains 1,000 records, 20 applicant features and one target column. The target is mapped to `0 = No Default` and `1 = Default`.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
columns = [
    'Status', 'Duration', 'CreditHistory', 'Purpose', 'CreditAmount',
    'Savings', 'Employment', 'InstallmentRate', 'PersonalStatus',
    'OtherDebtors', 'ResidenceSince', 'Property', 'Age',
    'OtherInstallmentPlans', 'Housing', 'ExistingCredits',
    'Job', 'Dependents', 'Telephone', 'ForeignWorker', 'Target'
]

df = pd.read_csv(url, sep=r"\\s+", names=columns)
df['Target'] = df['Target'].map({1: 0, 2: 1})

print("Dataset shape:", df.shape)
display(df.head())
print("\\nTarget distribution:")
print(df['Target'].value_counts().sort_index())

## 3. Prepare the data

Categorical columns are converted using One-Hot Encoding. A stratified split creates training, validation and test sets. The scaler is fitted only on the training data to prevent test-data leakage.

The final encoded feature count should be 48 for this dataset and encoding configuration.

In [ ]:
X = df.drop(columns=['Target'])
y = df['Target'].astype(np.int64)

categorical_cols = X.select_dtypes(include='object').columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype=float)

# First reserve 20% as a final, untouched test set.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=SEED, stratify=y
)

# Split the remaining 80% into training and validation portions.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20,
    random_state=SEED, stratify=y_train_full
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

feature_names = X_encoded.columns.tolist()
print("Encoded features:", len(feature_names))
print("Train:", X_train_scaled.shape, "Validation:", X_val_scaled.shape, "Test:", X_test_scaled.shape)

## 4. Convert data to PyTorch tensors

The model expects floating-point feature tensors and binary target tensors. A DataLoader supplies mini-batches during training.

In [ ]:
def make_loader(X_array, y_series, batch_size=32, shuffle=False):
    x_tensor = torch.tensor(np.asarray(X_array), dtype=torch.float32)
    y_tensor = torch.tensor(np.asarray(y_series), dtype=torch.float32).view(-1, 1)
    dataset = TensorDataset(x_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train_scaled, y_train, batch_size=32, shuffle=True)
val_loader = make_loader(X_val_scaled, y_val, batch_size=64, shuffle=False)

## 5. Define the Artificial Neural Network

- **Input layer:** one value for each of the 48 encoded features.
- **Hidden layer 1:** 64 neurons with ReLU activation.
- **Dropout:** randomly disables a fraction of activations during training to reduce overfitting.
- **Hidden layer 2:** 32 neurons with ReLU activation.
- **Output layer:** one logit for binary classification.

The network returns a logit. `BCEWithLogitsLoss` combines the sigmoid operation and binary cross-entropy in a numerically stable loss function.

In [ ]:
class CreditRiskANN(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.Dropout(p=0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p=0.20),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

model = CreditRiskANN(input_features=X_train_scaled.shape[1]).to(DEVICE)
print(model)

## 6. Train the network

The positive-class weight is calculated from the training split to account for the smaller default class. Adam updates the network weights using gradients computed by backpropagation. Early stopping monitors validation loss and restores the best model weights.

In [ ]:
num_negative = int((y_train == 0).sum())
num_positive = int((y_train == 1).sum())
pos_weight = torch.tensor([num_negative / max(num_positive, 1)], dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

def evaluate_loss(data_loader):
    model.eval()
    total_loss, total_count = 0.0, 0
    with torch.no_grad():
        for xb, yb in data_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(xb)
            total_count += len(xb)
    return total_loss / total_count

EPOCHS = 100
PATIENCE = 12
best_val_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0
history = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):
    model.train()
    running_loss, count = 0.0, 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)             # Forward propagation
        loss = criterion(logits, yb)   # Loss calculation
        loss.backward()                # Backpropagation
        optimizer.step()               # Adam parameter update

        running_loss += loss.item() * len(xb)
        count += len(xb)

    train_loss = running_loss / count
    val_loss = evaluate_loss(val_loader)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
print("Best validation loss:", round(best_val_loss, 4))

## 7. Visualise training and validation loss

A widening gap between training and validation loss can indicate overfitting. Early stopping helps limit unnecessary training.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="Training loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("ANN Training History")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 8. Evaluate on the untouched test set

The final test set is used only after training and model selection. The output logits are converted to probabilities using sigmoid. The default classification threshold is 0.50.

In [ ]:
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=DEVICE)
    test_logits = model(X_test_tensor).squeeze(1)
    y_prob_ann = torch.sigmoid(test_logits).cpu().numpy()

y_pred_ann = (y_prob_ann >= 0.50).astype(int)

print("ANN Accuracy:", round(accuracy_score(y_test, y_pred_ann), 4))
print("ANN ROC-AUC:", round(roc_auc_score(y_test, y_prob_ann), 4))
print("\\nClassification report:")
print(classification_report(y_test, y_pred_ann, target_names=["No Default", "Default"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred_ann)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Default", "Default"],
            yticklabels=["No Default", "Default"])
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("ANN Confusion Matrix (threshold = 0.50)")
plt.show()

## 9. Threshold comparison

In credit-risk screening, missing a defaulter may be costly. This cell compares thresholds on the final test probabilities for descriptive analysis only. For a rigorous model-selection workflow, select the threshold using validation-set predictions and evaluate it once on the test set.

In [ ]:
threshold_rows = []
for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    pred = (y_prob_ann >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="binary", zero_division=0
    )
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Accuracy": accuracy_score(y_test, pred)
    })

ann_threshold_results = pd.DataFrame(threshold_rows)
display(ann_threshold_results.round(4))

## 10. Compare with the existing ML baselines

This cell trains Logistic Regression and Random Forest using the same train/test partition and preprocessing data so that the comparison is easier to interpret. These models are conventional machine-learning baselines; the ANN is the deep-learning model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train_scaled, y_train)
lr_prob = lr.predict_proba(X_test_scaled)[:, 1]
lr_pred = (lr_prob >= 0.50).astype(int)

rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=SEED,
    class_weight="balanced"
)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.50).astype(int)

comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, lr_pred),
        "Default Recall": precision_recall_fscore_support(y_test, lr_pred, average="binary", zero_division=0)[1],
        "ROC-AUC": roc_auc_score(y_test, lr_prob)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, rf_pred),
        "Default Recall": precision_recall_fscore_support(y_test, rf_pred, average="binary", zero_division=0)[1],
        "ROC-AUC": roc_auc_score(y_test, rf_prob)
    },
    {
        "Model": "Artificial Neural Network",
        "Accuracy": accuracy_score(y_test, y_pred_ann),
        "Default Recall": precision_recall_fscore_support(y_test, y_pred_ann, average="binary", zero_division=0)[1],
        "ROC-AUC": roc_auc_score(y_test, y_prob_ann)
    }
])
display(comparison.round(4))

## 11. Save the trained model and preprocessing objects

The state dictionary contains the learned neural-network parameters. The scaler and encoded feature order are also saved because inference must use the same preprocessing and column order as training.

In [ ]:
import joblib
import json

torch.save(model.state_dict(), "credit_risk_ann.pth")
joblib.dump(scaler, "credit_risk_scaler.joblib")
joblib.dump(feature_names, "credit_risk_feature_names.joblib")

print("Saved: credit_risk_ann.pth")
print("Saved: credit_risk_scaler.joblib")
print("Saved: credit_risk_feature_names.joblib")

## 12. Key concepts for viva

1. **Neuron:** computes a weighted sum of inputs plus a bias.
2. **Weights and biases:** trainable parameters learned from data.
3. **ReLU:** introduces non-linearity in hidden layers.
4. **Sigmoid:** converts the output logit into a probability between 0 and 1.
5. **Forward propagation:** sends input features through the network to produce predictions.
6. **Binary cross-entropy:** measures the error between predicted probabilities and binary labels.
7. **Backpropagation:** calculates gradients of the loss with respect to model parameters.
8. **Adam optimizer:** updates weights using gradient-based optimisation.
9. **Dropout:** regularisation technique that reduces co-adaptation and can help prevent overfitting.
10. **Early stopping:** stops training when validation loss no longer improves.

## Conclusion

This notebook adds a genuine feed-forward ANN to the Credit Risk Prediction project. It demonstrates the basic deep-learning training cycle: forward propagation, loss calculation, backpropagation and parameter optimisation. The model's actual metrics must be generated by running all cells in order; do not write assumed ANN accuracy or claim that the ANN is better than the ML baselines before checking the results.